# AI Expense Tracker
Categorize spending, monitor budgets, flag unusual transactions, and export reports.

This local-first starter uses synthetic demo transactions. It does not need an API key. Keep real financial records private.

## 1. Install dependencies

In [ ]:
%pip install -q pandas numpy matplotlib scikit-learn

## 2. Imports

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

np.random.seed(42)
pd.set_option("display.max_columns", 50)

## 3. Create synthetic transactions

In [ ]:
rows = [
("2026-01-02","Monthly salary",52000,"Income"),("2026-01-03","Fresh vegetables and groceries",-1850,"Groceries"),
("2026-01-04","Metro card recharge",-600,"Transport"),("2026-01-05","Electricity bill",-1450,"Utilities"),
("2026-01-06","Apartment rent",-12000,"Housing"),("2026-01-08","Coffee and lunch",-420,"Dining"),
("2026-01-10","Online clothing order",-2499,"Shopping"),("2026-01-12","Pharmacy medicine",-680,"Healthcare"),
("2026-01-14","Movie tickets",-750,"Entertainment"),("2026-01-16","Fuel station",-1800,"Transport"),
("2026-01-18","Supermarket groceries",-2150,"Groceries"),("2026-01-20","Internet monthly bill",-999,"Utilities"),
("2026-01-23","Restaurant dinner",-1350,"Dining"),("2026-02-02","Monthly salary",52000,"Income"),
("2026-02-03","Grocery store",-2400,"Groceries"),("2026-02-04","Cab ride",-520,"Transport"),
("2026-02-05","Water and electricity bill",-1650,"Utilities"),("2026-02-06","Apartment rent",-12000,"Housing"),
("2026-02-08","Pizza delivery",-850,"Dining"),("2026-02-10","Headphones online purchase",-4999,"Shopping"),
("2026-02-12","Clinic consultation",-900,"Healthcare"),("2026-02-14","Streaming subscription",-499,"Entertainment"),
("2026-02-16","Petrol",-1600,"Transport"),("2026-02-18","Weekly groceries",-1950,"Groceries"),
("2026-02-21","Restaurant meal",-1120,"Dining"),("2026-03-02","Monthly salary",52000,"Income"),
("2026-03-03","Grocery store",-2200,"Groceries"),("2026-03-04","Bus pass",-750,"Transport"),
("2026-03-05","Electricity bill",-1520,"Utilities"),("2026-03-06","Apartment rent",-12000,"Housing"),
("2026-03-08","Cafe breakfast",-380,"Dining"),("2026-03-10","Emergency appliance repair",-9800,"Other"),
("2026-03-12","Pharmacy",-560,"Healthcare"),("2026-03-14","Concert ticket",-1800,"Entertainment"),
("2026-03-17","Online grocery order",-2050,"Groceries"),("2026-03-20","Mobile phone bill",-799,"Utilities"),
("2026-03-22","Fuel",-1700,"Transport")
]
df = pd.DataFrame(rows, columns=["date","description","amount","label"])
df["date"] = pd.to_datetime(df["date"])
df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
df = df.dropna(subset=["date","description","amount"]).copy()
df["transaction_type"] = np.where(df["amount"] >= 0, "Income", "Expense")
df["expense_amount"] = np.where(df["amount"] < 0, -df["amount"], 0.0)
df["month"] = df["date"].dt.to_period("M").astype(str)
display(df.head())

## 4. Explainable keyword categorization

In [ ]:
KEYWORDS = {
 "Groceries":["grocery","groceries","supermarket","vegetables","market"],
 "Dining":["restaurant","lunch","dinner","coffee","cafe","pizza","breakfast"],
 "Transport":["metro","bus","cab","taxi","fuel","petrol","ride"],
 "Housing":["rent","apartment","mortgage"],
 "Utilities":["electricity","water bill","internet","mobile phone bill"],
 "Shopping":["clothing","online purchase","headphones","shopping","order"],
 "Healthcare":["pharmacy","medicine","clinic","hospital","doctor","health"],
 "Entertainment":["movie","streaming","concert","game","theatre"],
 "Income":["salary","refund","interest income","cashback"]
}
def categorize(text):
    text = str(text).lower()
    for category, words in KEYWORDS.items():
        if any(word in text for word in words):
            return category
    return "Other"

df["rule_category"] = df["description"].apply(categorize)
display(df[["description","label","rule_category"]].head(12))

## 5. Train a small text classifier

In [ ]:
# This is a learning demo: the labeled synthetic dataset is small and not production-grade.
X_train, X_test, y_train, y_test = train_test_split(
    df["description"], df["label"], test_size=0.25, random_state=42, stratify=df["label"]
)
model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2))),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(classification_report(y_test, pred, zero_division=0))
df["predicted_category"] = model.predict(df["description"])
# Use supplied labels for the synthetic data; for new data, review predicted categories.
df["category"] = df["label"].fillna(df["rule_category"])
display(df[["description","label","predicted_category","category"]].head(12))

## 6. Overall and monthly summary

In [ ]:
income = df.loc[df.amount > 0, "amount"].sum()
expenses = df.loc[df.amount < 0, "expense_amount"].sum()
print(f"Total income:   {income:,.2f}")
print(f"Total expenses: {expenses:,.2f}")
print(f"Net cash flow:  {income-expenses:,.2f}")

expense_df = df[df.amount < 0].copy()
monthly = expense_df.groupby("month")["expense_amount"].sum().sort_index()
display(monthly.rename("expenses").to_frame())
monthly.plot(kind="bar", title="Monthly Expenses", xlabel="Month", ylabel="Amount")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 7. Spending by category

In [ ]:
category_spend = expense_df.groupby("category")["expense_amount"].sum().sort_values(ascending=False)
display(category_spend.rename("total_spend").to_frame())
category_spend.plot(kind="bar", title="Expenses by Category", xlabel="Category", ylabel="Amount")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

## 8. Budget monitoring

In [ ]:
budgets = {
 "Groceries":6000, "Dining":2500, "Transport":3000, "Shopping":4000,
 "Utilities":3500, "Entertainment":1500, "Healthcare":2500, "Housing":12000, "Other":2000
}
selected_month = expense_df["month"].max()
spent = expense_df[expense_df["month"] == selected_month].groupby("category")["expense_amount"].sum()
budget_report = pd.DataFrame([
 {"category": c, "budget": b, "actual": float(spent.get(c,0)),
  "remaining": float(b-spent.get(c,0)),
  "status": "OVER BUDGET" if spent.get(c,0) > b else "Within budget"}
 for c,b in budgets.items()
]).sort_values("remaining")
print("Budget report for", selected_month)
display(budget_report)

## 9. Flag unusually large expenses

In [ ]:
values = expense_df["expense_amount"]
median = values.median()
mad = (values - median).abs().median()
if mad > 0:
    scores = 0.6745 * (values - median).abs() / mad
    expense_df["unusual_flag"] = scores > 3.5
else:
    expense_df["unusual_flag"] = values > values.quantile(0.99)
review_queue = expense_df[expense_df["unusual_flag"]].sort_values("expense_amount", ascending=False)
display(review_queue[["date","description","expense_amount","category","unusual_flag"]])
print("These are review suggestions, not proof of fraud.")

## 10. Simple natural-language spending queries

In [ ]:
def answer_question(question):
    q = question.lower()
    if "income" in q:
        return f"Total income: {df.loc[df.amount > 0, 'amount'].sum():,.2f}"
    if "total expense" in q or "total spending" in q or "spent overall" in q:
        return f"Total expenses: {expense_df['expense_amount'].sum():,.2f}"
    if "highest" in q or "most spent" in q or "top category" in q:
        s = expense_df.groupby("category")["expense_amount"].sum().sort_values(ascending=False)
        return f"Highest category: {s.index[0]} ({s.iloc[0]:,.2f})"
    match = re.search(r"(groceries|dining|transport|housing|utilities|shopping|healthcare|entertainment|other)", q)
    if match:
        cat = match.group(1).title()
        total = expense_df.loc[expense_df.category.str.lower() == cat.lower(), "expense_amount"].sum()
        return f"Total spending for {cat}: {total:,.2f}"
    return "Try: 'total expenses', 'highest spending category', or 'spending on groceries'."

for q in ["What are my total expenses?", "Which category has the highest spending?", "How much did I spend on groceries?"]:
    print("Q:", q, "\nA:", answer_question(q), "\n")

## 11. Export reports

In [ ]:
output_dir = Path("expense_tracker_exports")
output_dir.mkdir(exist_ok=True)
df.to_csv(output_dir / "categorized_transactions.csv", index=False)
budget_report.to_csv(output_dir / "budget_report.csv", index=False)
review_queue.to_csv(output_dir / "unusual_expenses_review.csv", index=False)
print("Reports saved to:", output_dir.resolve())

## 12. Use your own CSV (optional)

Expected columns: `date`, `description`, `amount`. Positive amounts are treated as income and negative amounts as expenses. Keep the file local and private.

```python
my_df = pd.read_csv("my_sanitized_transactions.csv")
my_df["date"] = pd.to_datetime(my_df["date"], errors="coerce")
my_df["amount"] = pd.to_numeric(my_df["amount"], errors="coerce")
my_df = my_df.dropna(subset=["date", "description", "amount"])
my_df["category"] = my_df["description"].apply(categorize)
```

## Limitations and next steps
The model is trained on a small synthetic dataset, categories require review, anomaly flags are not fraud detection, and the query helper supports only a few patterns. Extend with CSV import validation, a Streamlit UI, recurring-payment detection, receipt OCR, user-corrected labels, and secure storage.